# Gold — Source Dimension
Static dimension describing each external data source feeding the pipeline.\
Included so indicator provenance is queryable without joining through the indicator dimension.

**Output:** `gold.dim_source` (Delta table)  
**Grain:** One row per data source

In [ ]:
import pandas as pd

sources = [
    (1, "SEDLABANKI", "Central Bank of Iceland", "Seðlabanki Íslands", "XML",  "XML",         "Daily / Monthly",     "https://sedlabanki.is"),
    (2, "HAGSTOFA",   "Statistics Iceland",      "Hagstofa Íslands",   "REST", "PX-Web JSON", "Monthly / Quarterly", "https://hagstofa.is"),
]

df_src = pd.DataFrame(sources, columns=[
    "source_key", "source_code", "source_name", "source_full_name",
    "api_type", "api_format", "update_frequency", "source_url"
])

In [ ]:
df_spark = spark.createDataFrame(df_src)
df_spark.createOrReplaceTempView("v_dim_source")

if not spark.catalog.tableExists("gold.dim_source"):
    df_spark.write.format("delta").saveAsTable("gold.dim_source")
else:
    spark.sql("""
        MERGE INTO gold.dim_source AS target
        USING v_dim_source AS source
        ON target.source_key = source.source_key
        WHEN MATCHED THEN UPDATE SET
            target.source_code      = source.source_code,
            target.source_name      = source.source_name,
            target.source_full_name = source.source_full_name,
            target.api_type         = source.api_type,
            target.api_format       = source.api_format,
            target.update_frequency = source.update_frequency,
            target.source_url       = source.source_url
        WHEN NOT MATCHED THEN INSERT (
            source_key, source_code, source_name, source_full_name,
            api_type, api_format, update_frequency, source_url
        )
        VALUES (
            source.source_key, source.source_code, source.source_name, source.source_full_name,
            source.api_type, source.api_format, source.update_frequency, source.source_url
        )
    """)

print(f"Dimension updated: {spark.table('gold.dim_source').count()} rows.")